In [1]:
import pickle
import pandas as pd

In [2]:
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/imb_files/hpt_results_working_data.pkl', 'rb') as f:
    hpt_results = pickle.load(f)

In [20]:
data_dict = {
    'mean': hpt_results['3s_no']['mean_scores'],
    'std': hpt_results['3s_no']['std_scores']
}

results_df = pd.DataFrame(data_dict).T
results_df

,accuracy,precision,recall,f1,f1_macro,f1_weighted
mean,0.724163,0.689991,0.667376,0.649948,0.708363,0.716685
std,0.065383,0.095545,0.232706,0.135995,0.083586,0.082516


In [21]:
data_dict = {
    'mean': hpt_results['5s_no']['mean_scores'],
    'std': hpt_results['5s_no']['std_scores']
}

results_df = pd.DataFrame(data_dict).T
results_df

,accuracy,precision,recall,f1,f1_macro,f1_weighted
mean,0.704442,0.644475,0.684044,0.650445,0.694572,0.701671
std,0.093460,0.128444,0.200311,0.136384,0.101940,0.096656


In [22]:
data_dict = {
    'mean': hpt_results['5s_0.5']['mean_scores'],
    'std': hpt_results['5s_0.5']['std_scores']
}

results_df = pd.DataFrame(data_dict).T
results_df

,accuracy,precision,recall,f1,f1_macro,f1_weighted
mean,0.691398,0.649356,0.634653,0.627947,0.679488,0.687443
std,0.054028,0.088577,0.160334,0.079218,0.055040,0.054732


In [23]:
data_dict = {
    'mean': hpt_results['2s_0.5']['mean_scores'],
    'std': hpt_results['2s_0.5']['std_scores']
}

results_df = pd.DataFrame(data_dict).T
results_df

,accuracy,precision,recall,f1,f1_macro,f1_weighted
mean,0.705185,0.652800,0.65821,0.640527,0.693714,0.703214
std,0.082527,0.074311,0.17073,0.081339,0.078905,0.088162


In [24]:
hpt_results['5s_0.5']

{'mean_scores': {'accuracy': 0.6913975805799574,
  'precision': 0.6493557965002328,
  'recall': 0.6346534825223349,
  'f1': 0.6279473876413493,
  'f1_macro': 0.6794881350981491,
  'f1_weighted': 0.6874431284352334},
 'std_scores': {'accuracy': 0.05402828269825749,
  'precision': 0.08857712537167688,
  'recall': 0.16033388036293358,
  'f1': 0.07921759181168178,
  'f1_macro': 0.05504023166850224,
  'f1_weighted': 0.05473200730298947},
 'individual_scores': [{'accuracy': 0.5971223021582733,
   'precision': 0.5423728813559322,
   'recall': 0.5245901639344263,
   'f1': 0.5333333333333333,
   'f1_macro': 0.5894514767932489,
   'f1_weighted': 0.5963148468566919},
  {'accuracy': 0.7007299270072993,
   'precision': 0.7878787878787878,
   'recall': 0.43333333333333335,
   'f1': 0.5591397849462365,
   'f1_macro': 0.666310223964831,
   'f1_weighted': 0.6796087455948756},
  {'accuracy': 0.7226277372262774,
   'precision': 0.6538461538461539,
   'recall': 0.6296296296296297,
   'f1': 0.6415094339622

#### Step 1 - Tabulate best configs per dataset

In [26]:
summary_tables = {}

for dataset_name, res in hpt_results.items():
    configs = pd.DataFrame(res['best_configs'])
    configs['fold'] = range(1, len(configs) + 1)

    # Count frequency of model types & imbalance handling
    model_counts = configs['model'].value_counts()
    imbalance_counts = configs['imbalance_technique'].value_counts()

    summary_tables[dataset_name] = {
        "configs_df": configs,
        "model_counts": model_counts,
        "imbalance_counts": imbalance_counts
    }
    
    

In [36]:
df = summary_tables['2s_0.5']
df['configs_df']
# Save dataframe to CSV


,model,imbalance_technique,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,early_stopping_rounds,min_samples_split,min_samples_leaf,max_features,bootstrap,fold
0,xgb,smote,98,3,0.087107,0.725742,0.803428,10.0,1.246461,0.820766,1.511102,30.0,NaN,NaN,NaN,NaN,1
1,rf,smote_tomek,398,15,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,11.0,4.0,log2,True,2
2,xgb,smote_tomek,287,3,0.103662,0.680985,0.913103,5.0,0.467524,1.168008,0.783209,76.0,NaN,NaN,NaN,NaN,3
3,rf,smote_tomek,368,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,17.0,9.0,log2,True,4
4,xgb,smote,460,3,0.102152,0.695080,0.648237,2.0,1.895101,1.523980,1.638624,42.0,NaN,NaN,NaN,NaN,5


#### Step 2 - Decide on the model type & imbalance handling 

We picked the most frequent choice across folds for that dataset

In [6]:
for dataset_name, info in summary_tables.items():
    most_common_model = info['model_counts'].idxmax()
    most_common_imbalance = info['imbalance_counts'].idxmax()
    print(f"{dataset_name}: Model → {most_common_model}, Imbalance → {most_common_imbalance}")

3s_no: Model → rf, Imbalance → smote_tomek
5s_no: Model → xgb, Imbalance → tomek_links
5s_0.5: Model → rf, Imbalance → smote
2s_0.5: Model → xgb, Imbalance → smote_tomek


#### Step 3 – Aggregate hyperparameters

For numeric hyperparameters (like n_estimators, max_depth) we averaged them across folds only for the winning model type 

In [7]:
final_choices = {}

for dataset_name, res in hpt_results.items():
    configs_df = pd.DataFrame(res['best_configs'])
    

    # Pick the most common model type for that dataset
    best_model = configs_df['model'].mode()[0]
    
    if best_model == 'rf':
        configs_df.drop(['learning_rate', 'subsample', 'min_child_weight', 'gamma', 'reg_alpha', 'reg_lambda', 'early_stopping_rounds', 'colsample_bytree'], axis=1, inplace=True)
    elif best_model == 'xgb':
        configs_df.drop(['min_samples_split', 'min_samples_leaf', 'max_features', 'bootstrap'], axis=1, inplace=True)
    
    best_imbalance = configs_df['imbalance_technique'].mode()[0]

    # Filter only configs with that model type
    best_model_configs = configs_df[configs_df['model'] == best_model]

    # Take median for stability
    # median_params = best_model_configs.median(numeric_only=True).astype(int).to_dict()
    median_params = best_model_configs.median(numeric_only=True).to_dict()
    
    # Add non-numeric parameters - only the ones that are relevant for the best model
    # XGBoost has no non-numeric parameters in this case
    if best_model == 'rf':
        median_params['max_features'] = best_model_configs['max_features'].mode()[0]
        median_params['bootstrap'] = best_model_configs['bootstrap'].mode()[0]

    final_choices[dataset_name] = {
        "model": best_model,
        "imbalance_technique": best_imbalance,
        "params": median_params
    }

In [7]:
# Save the final choices
with open('/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/imb_files/final_hpt_choices.pkl', 'wb') as f:
    pickle.dump(final_choices, f)   
    


In [9]:
res = hpt_results['5s_no']
# res['best_configs']

# Pick the most common model type for that dataset
configs_df = pd.DataFrame(res['best_configs'])


best_model = configs_df['model'].mode()[0]
best_imbalance = configs_df['imbalance_technique'].mode()[0]

if best_model == 'rf':
    configs_df.drop(['learning_rate', 'subsample', 'min_child_weight', 'gamma', 'reg_alpha', 'reg_lambda', 'early_stopping_rounds', 'colsample_bytree'], axis=1, inplace=True)
elif best_model == 'xgb':
    configs_df.drop(['min_samples_split', 'min_samples_leaf', 'max_features', 'bootstrap'], axis=1, inplace=True)
    
# Filter only configs with that model type
best_model_configs = configs_df[configs_df['model'] == best_model]
# Take median for stability
median_params = best_model_configs.median(numeric_only=True).to_dict()


if best_model == 'rf':
    median_params['max_features'] = best_model_configs['max_features'].mode()[0]
    median_params['bootstrap'] = best_model_configs['bootstrap'].mode()[0]


print(f"Best model for 3s_no: {best_model}, Imbalance technique: {best_imbalance}")
print(f"Median parameters: {median_params}")
best_model_configs

Best model for 3s_no: xgb, Imbalance technique: smote
Median parameters: {'n_estimators': 185.0, 'max_depth': 9.0, 'learning_rate': 0.2582394875217198, 'subsample': 0.6110759991262656, 'colsample_bytree': 0.9709924037567215, 'min_child_weight': 4.0, 'gamma': 1.2805909636800656, 'reg_alpha': 1.603483297512848, 'reg_lambda': 0.9287066172858303, 'early_stopping_rounds': 37.0}


,model,imbalance_technique,n_estimators,max_depth,learning_rate,subsample,colsample_bytree,min_child_weight,gamma,reg_alpha,reg_lambda,early_stopping_rounds
0,xgb,smote_tomek,212,10,0.258239,0.611076,0.970992,4.0,1.810998,1.208674,1.991623,11.0
1,xgb,tomek_links,135,9,0.283736,0.609027,0.973832,1.0,0.153598,1.603483,0.928707,98.0
4,xgb,smote,185,8,0.051901,0.733411,0.792328,6.0,1.280591,1.623402,0.350197,37.0


In [8]:
final_choices['5s_0.5']

{'model': 'rf',
 'imbalance_technique': 'smote',
 'params': {'n_estimators': 170.0,
  'max_depth': 18.0,
  'min_samples_split': 8.0,
  'min_samples_leaf': 9.0,
  'max_features': 'sqrt',
  'bootstrap': True}}

In [10]:
# hpt_results['5s_0.5']['best_configs']
hpt_results['5s_0.5']

{'mean_scores': {'accuracy': 0.6913975805799574,
  'precision': 0.6493557965002328,
  'recall': 0.6346534825223349,
  'f1': 0.6279473876413493,
  'f1_macro': 0.6794881350981491,
  'f1_weighted': 0.6874431284352334},
 'std_scores': {'accuracy': 0.05402828269825749,
  'precision': 0.08857712537167688,
  'recall': 0.16033388036293358,
  'f1': 0.07921759181168178,
  'f1_macro': 0.05504023166850224,
  'f1_weighted': 0.05473200730298947},
 'individual_scores': [{'accuracy': 0.5971223021582733,
   'precision': 0.5423728813559322,
   'recall': 0.5245901639344263,
   'f1': 0.5333333333333333,
   'f1_macro': 0.5894514767932489,
   'f1_weighted': 0.5963148468566919},
  {'accuracy': 0.7007299270072993,
   'precision': 0.7878787878787878,
   'recall': 0.43333333333333335,
   'f1': 0.5591397849462365,
   'f1_macro': 0.666310223964831,
   'f1_weighted': 0.6796087455948756},
  {'accuracy': 0.7226277372262774,
   'precision': 0.6538461538461539,
   'recall': 0.6296296296296297,
   'f1': 0.6415094339622

In [12]:
hpt_results['5s_0.5']['best_configs']

[{'model': 'xgb',
  'imbalance_technique': 'smote_tomek',
  'n_estimators': 469,
  'max_depth': 10,
  'learning_rate': 0.18310068625709486,
  'subsample': 0.7149924726805651,
  'colsample_bytree': 0.8780655916328991,
  'min_child_weight': 4,
  'gamma': 3.3064861282194236,
  'reg_alpha': 0.19636867704941746,
  'reg_lambda': 0.8225127502430364,
  'early_stopping_rounds': 90},
 {'model': 'rf',
  'imbalance_technique': 'smote',
  'n_estimators': 159,
  'max_depth': 18,
  'min_samples_split': 8,
  'min_samples_leaf': 2,
  'max_features': 'sqrt',
  'bootstrap': True},
 {'model': 'rf',
  'imbalance_technique': 'tomek_links',
  'n_estimators': 216,
  'max_depth': 14,
  'min_samples_split': 20,
  'min_samples_leaf': 10,
  'max_features': 'sqrt',
  'bootstrap': False},
 {'model': 'xgb',
  'imbalance_technique': 'smote',
  'n_estimators': 174,
  'max_depth': 4,
  'learning_rate': 0.06725963969331686,
  'subsample': 0.6860987545188013,
  'colsample_bytree': 0.7080706911536134,
  'min_child_weight'